In [10]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = "/usr/lib/jvm/java-17-openjdk-amd64/bin:" + os.environ["PATH"]

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ModelTraining") \
    .master("local[2]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

print("Spark Started Successfully ")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/03 08:26:29 WARN Utils: Your hostname, vasu, resolves to a loopback address: 127.0.0.1; using 10.255.255.254 instead (on interface lo)
26/03/03 08:26:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 08:26:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Started Successfully 


In [2]:
df = spark.read.parquet("data/processed/mobility_clean")
df.show(5)

+-------------------+----------+----------+-----------+--------------+----------------+--------------------+--------+-------+---------+---------+----------+-----------+-----------+------------+-------------+-------------+---------+-----------------+----+------+-----+
|               date|state_fips|state_code|county_fips|        county|pop_stay_at_home|pop_not_stay_at_home|   trips|trips_1|trips_1_3|trips_3_5|trips_5_10|trips_10_25|trips_25_50|trips_50_100|trips_100_250|trips_250_500|trips_500|           row_id|week| level|month|
+-------------------+----------+----------+-----------+--------------+----------------+--------------------+--------+-------+---------+---------+----------+-----------+-----------+------------+-------------+-------------+---------+-----------------+----+------+-----+
|2019-01-23 00:00:00|        37|        NC|      37117| Martin County|          3657.0|             18783.0| 84173.0|15360.0|  17558.0|  12451.0|   12006.0|    15879.0|     8012.0|      1771.0|   

In [4]:
from pyspark.sql.functions import col

df = df.withColumn(
    "stay_home_ratio",
    col("pop_stay_at_home") /
    (col("pop_stay_at_home") + col("pop_not_stay_at_home"))
)

print("Stay Home Ratio Added ")
df.select("stay_home_ratio").show(5)

Stay Home Ratio Added 
+-------------------+
|    stay_home_ratio|
+-------------------+
|0.16296791443850267|
|0.18639589169000934|
|0.17848832198569337|
| 0.1544378698224852|
|0.21463210201760163|
+-------------------+
only showing top 5 rows


In [6]:
df.explain(True)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(stay_home_ratio, '`/`('pop_stay_at_home, '`+`('pop_stay_at_home, 'pop_not_stay_at_home)), None)]
+- Relation [date#0,state_fips#1,state_code#2,county_fips#3,county#4,pop_stay_at_home#5,pop_not_stay_at_home#6,trips#7,trips_1#8,trips_1_3#9,trips_3_5#10,trips_5_10#11,trips_10_25#12,trips_25_50#13,trips_50_100#14,trips_100_250#15,trips_250_500#16,trips_500#17,row_id#18,week#19,level#20,month#21] parquet

== Analyzed Logical Plan ==
date: timestamp, state_fips: int, state_code: string, county_fips: int, county: string, pop_stay_at_home: double, pop_not_stay_at_home: double, trips: double, trips_1: double, trips_1_3: double, trips_3_5: double, trips_5_10: double, trips_10_25: double, trips_25_50: double, trips_50_100: double, trips_100_250: double, trips_250_500: double, trips_500: double, row_id: string, week: int, level: string, month: int, stay_home_ratio: double
Project [date#0, state_fips#1, state_code#2, county_fips#3, count

In [7]:
print("Columns After Feature Engineering:")
print(df.columns)

Columns After Feature Engineering:
['date', 'state_fips', 'state_code', 'county_fips', 'county', 'pop_stay_at_home', 'pop_not_stay_at_home', 'trips', 'trips_1', 'trips_1_3', 'trips_3_5', 'trips_5_10', 'trips_10_25', 'trips_25_50', 'trips_50_100', 'trips_100_250', 'trips_250_500', 'trips_500', 'row_id', 'week', 'level', 'month', 'stay_home_ratio']


In [8]:
try:
    df = spark.read.parquet("data/processed/mobility_clean")
    print("Data Loaded Successfully ")
except Exception as e:
    print("Error Loading Data:", str(e))

Data Loaded Successfully 


In [9]:
required_columns = ["trips", "pop_stay_at_home", "pop_not_stay_at_home"]

for col_name in required_columns:
    if col_name not in df.columns:
        raise ValueError(f"Missing Required Column: {col_name}")

print("All Required Columns Present ")

All Required Columns Present 


In [10]:
from pyspark.sql.functions import when

df = df.withColumn(
    "mobility_ratio",
    when(
        (df.pop_stay_at_home + df.pop_not_stay_at_home) != 0,
        df.pop_not_stay_at_home /
        (df.pop_stay_at_home + df.pop_not_stay_at_home)
    ).otherwise(0)
)

print("Division-by-zero handled safely ")

Division-by-zero handled safely 


In [11]:
if df.filter(df.trips < 0).count() > 0:
    print("Warning: Negative trips detected ⚠")
else:
    print("Trips validation passed ")

Trips validation passed 


In [13]:
import datetime

print("Transformation Timestamp:", datetime.datetime.now())
print("Total Rows After Validation:", df.count())
print("Total Columns:", len(df.columns))
print("Final Columns:", df.columns)

Transformation Timestamp: 2026-03-02 20:32:50.136571
Total Rows After Validation: 4280945
Total Columns: 23
Final Columns: ['date', 'state_fips', 'state_code', 'county_fips', 'county', 'pop_stay_at_home', 'pop_not_stay_at_home', 'trips', 'trips_1', 'trips_1_3', 'trips_3_5', 'trips_5_10', 'trips_10_25', 'trips_25_50', 'trips_50_100', 'trips_100_250', 'trips_250_500', 'trips_500', 'row_id', 'week', 'level', 'month', 'mobility_ratio']


In [14]:
print("DATA QUALITY SUMMARY ")
print("Row Count:", df.count())
print("Null Count Check:")
df.select([df[c].isNull().cast("int").alias(c) for c in df.columns]).groupBy().sum().show()

DATA QUALITY SUMMARY 
Row Count: 4280945
Null Count Check:


[Stage 15:======================================>                 (16 + 7) / 23]

+---------+---------------+---------------+----------------+-----------+---------------------+-------------------------+----------+------------+--------------+--------------+---------------+----------------+----------------+-----------------+------------------+------------------+--------------+-----------+---------+----------+----------+-------------------+
|sum(date)|sum(state_fips)|sum(state_code)|sum(county_fips)|sum(county)|sum(pop_stay_at_home)|sum(pop_not_stay_at_home)|sum(trips)|sum(trips_1)|sum(trips_1_3)|sum(trips_3_5)|sum(trips_5_10)|sum(trips_10_25)|sum(trips_25_50)|sum(trips_50_100)|sum(trips_100_250)|sum(trips_250_500)|sum(trips_500)|sum(row_id)|sum(week)|sum(level)|sum(month)|sum(mobility_ratio)|
+---------+---------------+---------------+----------------+-----------+---------------------+-------------------------+----------+------------+--------------+--------------+---------------+----------------+----------------+-----------------+------------------+------------------+

In [15]:
df.write \
  .mode("overwrite") \
  .parquet("data/final/mobility_ml_ready_v1")

26/03/02 20:33:30 WARN MemoryManager: Total allocation exceeds 95.00% (2,040,109,440 bytes) of heap memory
Scaling row group sizes to 95.00% for 16 writers
                                                                                

In [16]:
!ls data/final/

IOStream.flush timed out
mobility_ml_ready_v1


In [3]:
df = spark.read.parquet("data/final/mobility_ml_ready_v1")

print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 4280945
Columns: 23


In [4]:
print(df.columns)

['date', 'state_fips', 'state_code', 'county_fips', 'county', 'pop_stay_at_home', 'pop_not_stay_at_home', 'trips', 'trips_1', 'trips_1_3', 'trips_3_5', 'trips_5_10', 'trips_10_25', 'trips_25_50', 'trips_50_100', 'trips_100_250', 'trips_250_500', 'trips_500', 'row_id', 'week', 'level', 'month', 'mobility_ratio']


In [5]:
numeric_cols = [
    "pop_stay_at_home",
    "pop_not_stay_at_home",
    "mobility_ratio",
    "week",
    "month"
]

print("Selected Feature Columns:", numeric_cols)

Selected Feature Columns: ['pop_stay_at_home', 'pop_not_stay_at_home', 'mobility_ratio', 'week', 'month']


In [7]:
!pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 13.4 MB/s eta 0:00:0000:0100:01


In [8]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=numeric_cols,
    outputCol="features"
)

df_ml = assembler.transform(df)

print("Feature Vector Created ")
df_ml.select("features", "trips").show(5, truncate=False)

Feature Vector Created 
+--------------------------------------------+--------+
|features                                    |trips   |
+--------------------------------------------+--------+
|[3657.0,18783.0,0.8370320855614973,3.0,1.0] |84173.0 |
|[19963.0,87137.0,0.8136041083099906,3.0,1.0]|392257.0|
|[2071.0,9532.0,0.8215116780143067,2.0,1.0]  |51507.0 |
|[522.0,2858.0,0.8455621301775148,4.0,1.0]   |9490.0  |
|[3585.0,13118.0,0.7853678979823984,1.0,1.0] |59759.0 |
+--------------------------------------------+--------+
only showing top 5 rows


In [9]:
train, test = df_ml.randomSplit([0.8, 0.2], seed=42)

print("Training Rows:", train.count())
print("Testing Rows:", test.count())

Training Rows: 3424537


[Stage 10:======================================>                   (2 + 1) / 3]

Testing Rows: 856408


In [11]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    labelCol="trips",
    featuresCol="features"
)

model_lr = lr.fit(train)

pred_lr = model_lr.transform(test)

print("Linear Regression Training Completed ")

26/03/03 08:33:02 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/03 08:33:03 WARN Instrumentation: [05d65210] regParam is zero, which might cause numerical instability and overfitting.
26/03/03 08:33:08 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/03/03 08:33:13 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
[Stage 21:======================================>                   (2 + 1) / 3]

Linear Regression Training Completed 


In [12]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol="trips",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(pred_lr, {evaluator.metricName: "rmse"})
r2 = evaluator.evaluate(pred_lr, {evaluator.metricName: "r2"})

print("Linear Regression RMSE:", rmse)
print("Linear Regression R2:", r2)

[Stage 25:======================================>                   (2 + 1) / 3]

Linear Regression RMSE: 32370.49167993903
Linear Regression R2: 0.94564870080334


In [13]:
df.select("trips").describe().show()

+-------+------------------+
|summary|             trips|
+-------+------------------+
|  count|           4280945|
|   mean| 134257.1592973047|
| stddev|139055.64307214692|
|    min|               0.0|
|    max|          654879.0|
+-------+------------------+



In [14]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    labelCol="trips",
    featuresCol="features",
    numTrees=50,
    maxDepth=10
)

model_rf = rf.fit(train)

pred_rf = model_rf.transform(test)

print("Random Forest Training Completed ")

26/03/03 08:36:06 WARN MemoryStore: Not enough space to cache rdd_124_1 in memory! (computed 67.5 MiB so far)
26/03/03 08:36:06 WARN BlockManager: Persisting block rdd_124_1 to disk instead.
26/03/03 08:36:06 WARN MemoryStore: Not enough space to cache rdd_124_0 in memory! (computed 67.5 MiB so far)
26/03/03 08:36:06 WARN BlockManager: Persisting block rdd_124_0 to disk instead.
26/03/03 08:36:12 WARN MemoryStore: Not enough space to cache rdd_124_0 in memory! (computed 151.9 MiB so far)
26/03/03 08:36:12 WARN MemoryStore: Not enough space to cache rdd_124_1 in memory! (computed 228.0 MiB so far)
26/03/03 08:36:22 WARN MemoryStore: Not enough space to cache rdd_124_2 in memory! (computed 360.0 MiB so far)
26/03/03 08:36:22 WARN BlockManager: Persisting block rdd_124_2 to disk instead.
26/03/03 08:36:25 WARN MemoryStore: Not enough space to cache rdd_124_2 in memory! (computed 360.0 MiB so far)
26/03/03 08:36:27 WARN MemoryStore: Not enough space to cache rdd_124_0 in memory! (computed 

Random Forest Training Completed 


In [15]:
rmse_rf = evaluator.evaluate(pred_rf, {evaluator.metricName: "rmse"})
r2_rf = evaluator.evaluate(pred_rf, {evaluator.metricName: "r2"})

print("Random Forest RMSE:", rmse_rf)
print("Random Forest R2:", r2_rf)

[Stage 56:======================================>                   (2 + 1) / 3]

Random Forest RMSE: 29717.281310596914
Random Forest R2: 0.9541932490478142


In [16]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    labelCol="trips",
    featuresCol="features",
    maxIter=50,
    maxDepth=5
)

model_gbt = gbt.fit(train)

pred_gbt = model_gbt.transform(test)

print("Gradient Boosted Trees Training Completed ")

26/03/03 08:46:24 WARN MemoryStore: Not enough space to cache rdd_199_0 in memory! (computed 43.6 MiB so far)
26/03/03 08:46:24 WARN BlockManager: Persisting block rdd_199_0 to disk instead.
                                                                                

Gradient Boosted Trees Training Completed 


In [17]:
rmse_gbt = evaluator.evaluate(pred_gbt, {evaluator.metricName: "rmse"})
r2_gbt = evaluator.evaluate(pred_gbt, {evaluator.metricName: "r2"})

print("GBT RMSE:", rmse_gbt)
print("GBT R2:", r2_gbt)

[Stage 564:======================================>                  (2 + 1) / 3]

GBT RMSE: 29919.278066624436
GBT R2: 0.9535684097835152


In [18]:
from pyspark.ml.regression import DecisionTreeRegressor

dt = DecisionTreeRegressor(
    labelCol="trips",
    featuresCol="features",
    maxDepth=10
)

model_dt = dt.fit(train)

pred_dt = model_dt.transform(test)

print("Decision Tree Training Completed ")

26/03/03 08:52:15 WARN MemoryStore: Not enough space to cache rdd_1338_0 in memory! (computed 46.3 MiB so far)
26/03/03 08:52:15 WARN BlockManager: Persisting block rdd_1338_0 to disk instead.
26/03/03 08:52:15 WARN MemoryStore: Not enough space to cache rdd_1338_1 in memory! (computed 104.2 MiB so far)
26/03/03 08:52:15 WARN BlockManager: Persisting block rdd_1338_1 to disk instead.
26/03/03 08:52:17 WARN MemoryStore: Not enough space to cache rdd_1338_0 in memory! (computed 148.8 MiB so far)
26/03/03 08:52:20 WARN MemoryStore: Not enough space to cache rdd_1338_2 in memory! (computed 46.3 MiB so far)
26/03/03 08:52:20 WARN BlockManager: Persisting block rdd_1338_2 to disk instead.
26/03/03 08:52:22 WARN MemoryStore: Not enough space to cache rdd_1338_0 in memory! (computed 148.8 MiB so far)
26/03/03 08:52:23 WARN MemoryStore: Not enough space to cache rdd_1338_0 in memory! (computed 148.8 MiB so far)
26/03/03 08:52:24 WARN MemoryStore: Not enough space to cache rdd_1338_0 in memory! 

Decision Tree Training Completed 


In [19]:
rmse_dt = evaluator.evaluate(pred_dt, {evaluator.metricName: "rmse"})
r2_dt = evaluator.evaluate(pred_dt, {evaluator.metricName: "r2"})

print("Decision Tree RMSE:", rmse_dt)
print("Decision Tree R2:", r2_dt)

[Stage 592:======================================>                  (2 + 1) / 3]

Decision Tree RMSE: 30112.04430484163
Decision Tree R2: 0.9529681762976978


In [20]:
print("===== MODEL COMPARISON =====")

print("Linear Regression → RMSE:", rmse, "| R2:", r2)
print("Random Forest     → RMSE:", rmse_rf, "| R2:", r2_rf)
print("Gradient Boosted  → RMSE:", rmse_gbt, "| R2:", r2_gbt)
print("Decision Tree     → RMSE:", rmse_dt, "| R2:", r2_dt)

===== MODEL COMPARISON =====
Linear Regression → RMSE: 32370.49167993903 | R2: 0.94564870080334
Random Forest     → RMSE: 29717.281310596914 | R2: 0.9541932490478142
Gradient Boosted  → RMSE: 29919.278066624436 | R2: 0.9535684097835152
Decision Tree     → RMSE: 30112.04430484163 | R2: 0.9529681762976978


In [21]:
model_lr.write().overwrite().save("models/linear_regression_model")
print("Linear Regression Model Saved ")

Linear Regression Model Saved 


In [22]:
model_rf.write().overwrite().save("models/random_forest_model")
print("Random Forest Model Saved ")

26/03/03 08:59:42 WARN TaskSetManager: Stage 600 contains a task of very large size (9404 KiB). The maximum recommended task size is 1000 KiB.


Random Forest Model Saved 


In [23]:
model_gbt.write().overwrite().save("models/gbt_model")
print("GBT Model Saved ")

GBT Model Saved 


In [24]:
model_dt.write().overwrite().save("models/decision_tree_model")
print("Decision Tree Model Saved ")

Decision Tree Model Saved 


In [25]:
test.write.mode("overwrite").parquet("data/final/test_dataset")
print("Test Dataset Saved ")

[Stage 614:======================================>                  (2 + 1) / 3]

Test Dataset Saved 
